In [2]:
import pandas as pd
import random

# 配置基础数据池
platforms = ['京东', '淘宝']
categories = ['真无线耳机', '头戴式耳机', '运动蓝牙耳机', '有线HiFi耳机']
brands = ['Apple', '华为', '小米', '漫步者', '索尼', 'Bose', 'OPPO', 'vivo', '倍思', '绿联', 'QCY', 'SANAG']
features = ['主动降噪', '超长续航', '无线蓝牙5.3', 'HIFI音质', '入耳式', '半入耳', '低延迟游戏模式', '防水防汗', '空间音频']

data = []
total_records = 20000

print(f"开始生成 {total_records} 条数据，请稍候...")

for _ in range(total_records):
    platform = random.choice(platforms)
    category = random.choice(categories)
    
    # 模拟品牌效应：部分品牌偏高端，部分偏中低端
    if random.random() < 0.1: # 10% 是高端品牌
        brand = random.choice(['Apple', '索尼', 'Bose'])
        base_price = random.randint(800, 2500)
    elif random.random() < 0.4: # 30% 是国产头部
        brand = random.choice(['华为', '小米', '漫步者', 'OPPO', 'vivo'])
        base_price = random.randint(150, 800)
    else: # 60% 是白牌或入门款
        brand = random.choice(['倍思', '绿联', 'QCY', 'SANAG', '其他白牌'])
        base_price = random.randint(29, 200)
        
    # 店铺类型
    if platform == '京东':
        shop_type = random.choices(['京东自营', '官方旗舰店', '第三方专卖店'], weights=[0.5, 0.3, 0.2])[0]
    else:
        shop_type = random.choices(['天猫官方旗舰店', '淘宝C店', '天猫专卖店'], weights=[0.4, 0.4, 0.2])[0]

    # 计算折扣与价格
    original_price = base_price
    # 模拟大促：有些打折狠，有些不太打折
    discount = round(random.uniform(0.5, 0.95), 2) if random.random() < 0.7 else round(random.uniform(0.85, 1.0), 2)
    price = round(original_price * discount, 2)
    
    # 模拟销量：价格越低，销量基数越大，并加入随机波动
    sales_base = int(300000 / (price/10 + 1)) # 价格越低，基数越大
    sales = max(0, sales_base + random.randint(-2000, 5000))
    
    # 评价数（一般是销量的 10% - 30%）
    review_count = int(sales * random.uniform(0.1, 0.3))
    
    # 拼装商品标题
    f1, f2 = random.sample(features, 2)
    prefix = random.choice(['新款', '旗舰', '爆款', '2024', '官方正品'])
    title = f"{brand} {prefix} {category} {f1} {f2} {random.choice(['真无线', '头戴式', '运动', '入耳'])}"
    
    data.append({
        'platform': platform,
        'category': category,
        'brand': brand,
        'shop_type': shop_type,
        'title': title,
        'original_price': original_price,
        'price': price,
        'discount_rate': discount,
        'sales': sales,
        'review_count': review_count
    })

# 转成 DataFrame 并导出
df = pd.DataFrame(data)
df.to_csv('mock_ecommerce_data_20k.csv', index=False, encoding='utf-8-sig')

print(f"✅ 生成完毕！成功创建 {len(df)} 条数据。")
print(f"✅ 文件已保存为：mock_ecommerce_data_20k.csv")

开始生成 20000 条数据，请稍候...
✅ 生成完毕！成功创建 20000 条数据。
✅ 文件已保存为：mock_ecommerce_data_20k.csv


In [3]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# ---------------- 1. 用户维表 (dim_user) ----------------
num_users = 2000
user_ids = [f"U{i:05d}" for i in range(1, num_users + 1)]
users = pd.DataFrame({
    'user_id': user_ids,
    'register_date': [datetime(2024, 8, 1) + timedelta(days=random.randint(0, 90)) for _ in range(num_users)],
    'member_level': np.random.choice(['普通会员', '白银', '黄金', '铂金'], num_users, p=[0.6, 0.2, 0.15, 0.05]),
    'city_tier': np.random.choice(['一线', '新一线', '二线', '三线及以下'], num_users, p=[0.1, 0.2, 0.3, 0.4])
})
users.to_csv('dim_user.csv', index=False, encoding='utf-8-sig')

# ---------------- 2. 商品维表 (dim_product) ----------------
num_products = 100
categories = ['手机数码', '电脑办公', '家用电器', '美妆护肤', '食品饮料']
brands = ['Apple', '华为', '小米', '美的', '欧莱雅', '三只松鼠', '白牌']
products = pd.DataFrame({
    'sku_id': [f"SKU{i:04d}" for i in range(1, num_products + 1)],
    'category': np.random.choice(categories, num_products),
    'brand': np.random.choice(brands, num_products),
    'original_price': np.random.randint(50, 5000, num_products),
    'is_online': np.random.choice([1, 0], num_products, p=[0.9, 0.1]), # 90% 在线
    'is_exclusive': np.random.choice([1, 0], num_products, p=[0.1, 0.9]) # 10% 独家产品
})
products['current_price'] = products['original_price'] * np.random.uniform(0.6, 1.0, num_products).round(2)
products.to_csv('dim_product.csv', index=False, encoding='utf-8-sig')

# ---------------- 3. 流量日志表 (fact_traffic) ----------------
# 覆盖双11大促期间 (2024-11-01 至 2024-11-11)
num_traffic = 50000
start_date = datetime(2024, 11, 1)
traffic_data = []
for _ in range(num_traffic):
    # 大促当天流量爆发
    day_offset = random.choices(range(11), weights=[1,1,1,2,2,3,4,5,8,15,30])[0]
    visit_time = start_date + timedelta(days=day_offset, hours=random.randint(0,23), minutes=random.randint(0,59))
    source = np.random.choice(['自然搜索', '直通车广告', '站外抖音', '老客直接访问'], p=[0.4, 0.3, 0.2, 0.1])
    is_bounce = np.random.choice([1, 0], p=[0.4, 0.6]) # 跳出率 40%
    stay_time = random.randint(1, 10) if is_bounce else random.randint(30, 600) # 跳出则停留短
    
    traffic_data.append({
        'log_id': f"LOG{_+1:06d}",
        'user_id': random.choice(user_ids),
        'visit_time': visit_time,
        'source_channel': source,
        'is_bounce': is_bounce,
        'duration_sec': stay_time,
        'page_views': 1 if is_bounce else random.randint(2, 15)
    })
traffic = pd.DataFrame(traffic_data)
traffic.to_csv('fact_traffic.csv', index=False, encoding='utf-8-sig')

# ---------------- 4. 订单事实表 (fact_order) ----------------
num_orders = 15000
order_data = []
for i in range(num_orders):
    sku = products.sample(1).iloc[0]
    user = random.choice(user_ids)
    order_time = start_date + timedelta(days=random.choices(range(11), weights=[1,1,2,2,3,4,5,8,10,20,50])[0], hours=random.randint(0,23))
    
    # 模拟转化漏斗：加入购物车 -> 下单 -> 支付
    status_rand = random.random()
    if status_rand < 0.1:
        status = '加购未下单'
        pay_amount = 0
        pay_time = pd.NaT
    elif status_rand < 0.2:
        status = '下单未支付'
        pay_amount = 0
        pay_time = pd.NaT
    elif status_rand < 0.25:
        status = '已退款'
        pay_amount = sku['current_price']
        pay_time = order_time + timedelta(minutes=random.randint(1, 60))
    else:
        status = '交易成功'
        pay_amount = sku['current_price']
        pay_time = order_time + timedelta(minutes=random.randint(1, 60))
        
    # 随机分配营销活动
    campaign = np.random.choice(['无', '双11满减', '百亿补贴', '品牌特卖'], p=[0.4, 0.4, 0.1, 0.1])

    order_data.append({
        'order_id': f"ORD{i+1:07d}",
        'user_id': user,
        'sku_id': sku['sku_id'],
        'category': sku['category'],
        'order_time': order_time,
        'pay_time': pay_time,
        'order_status': status,
        'pay_amount': round(pay_amount, 2),
        'campaign': campaign
    })
orders = pd.DataFrame(order_data)
orders.to_csv('fact_order.csv', index=False, encoding='utf-8-sig')

# ---------------- 5. 评价风控表 (fact_review) ----------------
# 从交易成功的订单里抽 3000 条生成评价
success_orders = orders[orders['order_status'] == '交易成功'].sample(3000)
review_data = []
for idx, row in success_orders.iterrows():
    score = np.random.choice([5, 4, 3, 2, 1], p=[0.6, 0.2, 0.1, 0.05, 0.05])
    is_complaint = 1 if score <= 2 and random.random() < 0.3 else 0
    content = "物流很快，质量很好" if score >= 4 else ("有点小瑕疵，勉强能用" if score == 3 else "质量太差，包装破损，要求退款！")
    
    review_data.append({
        'review_id': f"REV{idx+1:06d}",
        'order_id': row['order_id'],
        'sku_id': row['sku_id'],
        'score': score,
        'review_content': content,
        'is_complaint': is_complaint
    })
reviews = pd.DataFrame(review_data)
reviews.to_csv('fact_review.csv', index=False, encoding='utf-8-sig')

print("✅ 全套电商数据生成完毕！(5个CSV文件已导出)")
print(f"用户数: {len(users)} | 商品数: {len(products)}")
print(f"流量日志: {len(traffic)} 条 | 订单: {len(orders)} 条 | 评价: {len(reviews)} 条")

✅ 全套电商数据生成完毕！(5个CSV文件已导出)
用户数: 2000 | 商品数: 100
流量日志: 50000 条 | 订单: 15000 条 | 评价: 3000 条
